# 01 — Data Generation
## Solvix E-commerce Analytics

**Objective:** Generate synthetic but realistic sales and marketing datasets that simulate 6 months of operations for Solvix, a Colombian dropshipping company.

**Why synthetic data?**  
Real business data is confidential. Generating synthetic data with embedded business logic allows us to practice a full analytics workflow while keeping the scenario credible and explainable.

**Outputs:**
- `data/raw/solvix_ordenes_raw.csv` — 3,500 orders across 5 Colombian cities
- `data/raw/solvix_meta_ads_raw.csv` — 180 days of Meta Ads campaign performance

**Period covered:** November 1, 2025 → April 30, 2026

## 1. Libraries

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

# Fixed seed — ensures reproducibility across runs
random.seed(42)
np.random.seed(42)

print("Libraries loaded successfully.")

## 2. Product Catalog

Solvix operates with 4 products across 3 categories. Key business decision: the **Mini Bike** has the highest ticket price ($120) but also the highest shipping cost due to weight (5 kg), which significantly compresses its margin.

Each product has a `prob_devolucion` (return probability) based on realistic e-commerce benchmarks:
- Low-weight accessories → low return rate (4–8%)
- High-ticket, heavy products → higher return rate (18%)

In [ ]:
productos = {
    'P001': {'nombre': 'Vaso Térmico Inteligente Auto', 'precio': 39,  'cogs': 15, 'peso_kg': 0.5, 'categoria': 'Auto',        'prob_devolucion': 0.05},
    'P002': {'nombre': 'Mini Bicicleta Premium',         'precio': 120, 'cogs': 45, 'peso_kg': 5.0, 'categoria': 'Fitness',     'prob_devolucion': 0.18},
    'P003': {'nombre': 'Soporte Magnético Carga',        'precio': 24,  'cogs': 8,  'peso_kg': 0.2, 'categoria': 'Auto',        'prob_devolucion': 0.04},
    'P004': {'nombre': 'Cojín Ergonómico Gel',           'precio': 34,  'cogs': 12, 'peso_kg': 1.2, 'categoria': 'Oficina/Auto','prob_devolucion': 0.08},
}

# Quick overview
df_catalog = pd.DataFrame(productos).T
df_catalog['margen_bruto_%'] = ((df_catalog['precio'].astype(float) - df_catalog['cogs'].astype(float)) / df_catalog['precio'].astype(float) * 100).round(1)
print(df_catalog[['nombre','precio','cogs','margen_bruto_%','peso_kg','prob_devolucion']])

## 3. Customer Pool

We create 2,000 unique customers for 3,500 orders, meaning some customers purchase more than once. Customer frequency follows a **Zipf distribution** — a small group of loyal customers places significantly more orders than average, which is realistic for e-commerce.

In [ ]:
num_clientes = 2000
ids_clientes = [f"CLI-{5000 + i}" for i in range(num_clientes)]

# Zipf distribution: top customers appear more frequently
pesos_clientes = np.array([1 / (i + 1) ** 0.6 for i in range(num_clientes)])
pesos_clientes = pesos_clientes / pesos_clientes.sum()

print(f"Customer pool: {num_clientes:,} unique customers")
print(f"Top 10% of customers account for ~{pesos_clientes[:200].sum():.1%} of all orders")

## 4. Acquisition Sources

Each order has an acquisition source linked to real Meta Ads campaigns or organic traffic. This column is the bridge between the orders table and the ads table — enabling ROAS and CPA calculations.

In [ ]:
fuentes = {
    'Campaña_Vaso_Vertical':  {'productos': ['P001', 'P003'], 'prob': 0.55},
    'Campaña_Bici_Carrusel':  {'productos': ['P002'],         'prob': 0.20},
    'Orgánico_Instagram':     {'productos': ['P001', 'P004'], 'prob': 0.15},
    'Directo':                {'productos': ['P001', 'P002', 'P003', 'P004'], 'prob': 0.10},
}
nombres_fuentes = list(fuentes.keys())
probs_fuentes   = [fuentes[f]['prob'] for f in nombres_fuentes]

print("Acquisition sources configured:")
for f, p in zip(nombres_fuentes, probs_fuentes):
    print(f"  {f}: {p:.0%} of orders")

## 5. Orders Generation

**Business logic embedded:**
- Bogotá and Medellín have higher demand for auto accessories (urban mobility)
- Smaller cities lean toward fitness and comfort products
- Shipping cost scales with product weight (heavier = less margin)
- Returns are product-specific, not uniform
- `Ganancia_Bruta` on returned orders is negative (lost COGS + shipping)

In [ ]:
num_ordenes   = 3500
ciudades      = ['Bogotá', 'Medellín', 'Cali', 'Barranquilla', 'Bucaramanga']
prob_ciudades = [0.40, 0.25, 0.15, 0.10, 0.10]
fecha_inicio  = datetime(2025, 11, 1)

ordenes = []

for i in range(num_ordenes):
    id_orden   = f"ORD-{1000 + i}"
    fecha      = fecha_inicio + timedelta(days=random.randint(0, 180), minutes=random.randint(0, 1440))
    id_cliente = np.random.choice(ids_clientes, p=pesos_clientes)
    ciudad     = np.random.choice(ciudades, p=prob_ciudades)
    fuente     = np.random.choice(nombres_fuentes, p=probs_fuentes)

    # Product probability biased by city and campaign
    fuente_productos = fuentes[fuente]['productos']
    if ciudad in ['Bogotá', 'Medellín']:
        candidatos = ['P001', 'P003']
        prob_base  = [0.55, 0.45]
    else:
        candidatos = ['P002', 'P004']
        prob_base  = [0.60, 0.40]

    if fuente_productos[0] in candidatos:
        idx = candidatos.index(fuente_productos[0])
        prob_base[idx] = min(prob_base[idx] + 0.20, 0.95)
        total = sum(prob_base)
        prob_base = [p / total for p in prob_base]
    else:
        candidatos = list(productos.keys())
        prob_base  = [0.30, 0.25, 0.25, 0.20]

    id_prod       = np.random.choice(candidatos, p=prob_base)
    prod          = productos[id_prod]
    cantidad      = np.random.choice([1, 2], p=[0.85, 0.15])
    ingreso_total = prod['precio'] * cantidad
    cogs_total    = prod['cogs']  * cantidad
    costo_envio   = round(random.uniform(3, 6) + prod['peso_kg'] * 1.5, 2)
    ganancia_bruta = round(ingreso_total - cogs_total - costo_envio, 2)

    # Status with product-specific return rate
    if random.random() < prod['prob_devolucion']:
        estado         = 'Devuelto'
        ganancia_bruta = round(-cogs_total - costo_envio, 2)
    elif random.random() < 0.12:
        estado = 'En tránsito'
    else:
        estado = 'Entregado'

    ordenes.append([
        id_orden, fecha.strftime('%Y-%m-%d %H:%M:%S'), id_cliente,
        id_prod, prod['nombre'], prod['categoria'],
        cantidad, ingreso_total, cogs_total, costo_envio, ganancia_bruta,
        ciudad, fuente, estado,
    ])

df_ordenes = pd.DataFrame(ordenes, columns=[
    'ID_Orden','Fecha','ID_Cliente','ID_Producto','Nombre_Producto','Categoria',
    'Cantidad','Ingreso_Total','COGS_Total','Costo_Envio','Ganancia_Bruta',
    'Ciudad_Destino','Fuente_Adquisicion','Estado_Envio',
])

print(f"Orders generated: {len(df_ordenes):,}")
print(f"Unique customers: {df_ordenes['ID_Cliente'].nunique():,}")
print(f"\nStatus distribution:")
print(df_ordenes['Estado_Envio'].value_counts(normalize=True).map('{:.1%}'.format))

## 6. Meta Ads Generation

**Seasonality logic:**
- Nov 15 – Dec 31: spend scales 1.6x–2.4x (Black Friday + Christmas peak)
- January: spend drops 0.6x–0.85x (post-holiday slump)
- Rest of period: normal fluctuation (0.9x–1.1x)

In [ ]:
ads_data = []

for dia in range(180):
    fecha_ad = fecha_inicio + timedelta(days=dia)
    dia_abs  = dia

    if 14 <= dia_abs <= 60:       # Black Friday → Christmas
        factor = random.uniform(1.6, 2.4)
    elif 61 <= dia_abs <= 75:     # January slump
        factor = random.uniform(0.6, 0.85)
    else:
        factor = random.uniform(0.9, 1.1)

    # Campaign: Vaso Vertical — high CTR, solid conversion
    ads_data.append([
        fecha_ad.strftime('%Y-%m-%d'), 'Campaña_Vaso_Vertical',
        round(random.uniform(15, 25) * factor, 2),
        int(random.randint(150, 300) * factor),
        int(random.randint(3, 8) * factor),
    ])

    # Campaign: Bici Carrusel — high spend, low volume, high ticket
    ads_data.append([
        fecha_ad.strftime('%Y-%m-%d'), 'Campaña_Bici_Carrusel',
        round(random.uniform(20, 35) * factor, 2),
        int(random.randint(80, 150) * factor),
        int(random.randint(1, 3) * factor),
    ])

df_ads = pd.DataFrame(ads_data, columns=[
    'Fecha','Nombre_Campaña','Gasto_Diario_USD','Clics','Compras_Atribuidas'
])

print(f"Ad records generated: {len(df_ads):,}")
print(f"Total ad spend:      ${df_ads['Gasto_Diario_USD'].sum():,.0f} USD")
print(f"Total attr. purchases: {df_ads['Compras_Atribuidas'].sum():,}")
print(f"\nSpend by campaign:")
print(df_ads.groupby('Nombre_Campaña')['Gasto_Diario_USD'].sum().map('${:,.0f}'.format))

## 7. Export to CSV

In [ ]:
output_dir = '../data/raw'
os.makedirs(output_dir, exist_ok=True)

df_ordenes.to_csv(f'{output_dir}/solvix_ordenes_raw.csv', index=False)
df_ads.to_csv(f'{output_dir}/solvix_meta_ads_raw.csv',    index=False)

print("Files exported successfully:")
print(f"  → {output_dir}/solvix_ordenes_raw.csv  ({len(df_ordenes):,} rows)")
print(f"  → {output_dir}/solvix_meta_ads_raw.csv ({len(df_ads):,} rows)")

## 8. Sanity Check

Quick validation to confirm the datasets are consistent before moving to cleaning and EDA.

In [ ]:
print("=" * 50)
print("ORDERS — SANITY CHECK")
print("=" * 50)
print(f"Total orders:         {len(df_ordenes):,}")
print(f"Unique customers:     {df_ordenes['ID_Cliente'].nunique():,}")
print(f"Date range:           {df_ordenes['Fecha'].min()} → {df_ordenes['Fecha'].max()}")
print(f"Total revenue:       ${df_ordenes.loc[df_ordenes['Estado_Envio']=='Entregado','Ingreso_Total'].sum():,.0f}")
print(f"Total gross profit:  ${df_ordenes['Ganancia_Bruta'].sum():,.0f}")
print(f"Overall return rate:  {(df_ordenes['Estado_Envio']=='Devuelto').mean():.1%}")

print(f"\nReturn rate by product:")
ret = df_ordenes.groupby('Nombre_Producto')['Estado_Envio'].apply(
    lambda x: (x == 'Devuelto').mean()
).round(3)
print(ret.to_string())

print(f"\n{'=' * 50}")
print("META ADS — SANITY CHECK")
print("=" * 50)
print(f"Total records:        {len(df_ads):,}")
print(f"Total spend:         ${df_ads['Gasto_Diario_USD'].sum():,.0f} USD")
print(f"Total attr. sales:    {df_ads['Compras_Atribuidas'].sum():,}")
print(f"\nAvg daily spend by campaign:")
print(df_ads.groupby('Nombre_Campaña')['Gasto_Diario_USD'].mean().round(2).to_string())